# LocalFood AI — Agentic AI demonstrations

This notebook runs the same Python tools and explicit plan-act-observe-decide loop used by the web app. Every section prints the evidence needed for a viva.

In [ ]:
import json
import os
import sys
from pprint import pprint

sys.path.insert(0, os.path.abspath("backend"))
from agent import run_agent
from memory import clear_memory, get_memory

def show(label, value):
    print(f"\n{'=' * 12} {label} {'=' * 12}")
    if isinstance(value, (dict, list)):
        pprint(value, sort_dicts=False)
    else:
        print(value)

def show_trace(result):
    for step in result["trace"]:
        print(f"STEP {step['step']} | {step['title']} | {step['detail']}")
        if step.get("tool"):
            print("  TOOL CALL:", step["tool"], step.get("arguments"))
            print("  TOOL RESULT:", step.get("result"))
    print("FINAL RESULT:", result["reply"])

print("LocalFood AI notebook ready")

## Demo 1 — full recommendation

The agent extracts constraints, writes memory, calls `find_restaurants`, observes the result, calls `filter_by_cuisine`, applies the vegetarian guardrail, and ranks the safe candidates.

In [ ]:
session = "notebook-demo-1"
clear_memory(session)
user_input = "I'm vegetarian and I want spicy Punjabi food in Jalandhar."
show("USER INPUT", user_input)
result = run_agent(session, user_input)
show("MEMORY READ/WRITE", result["memory"])
show("AGENT TRACE", result["trace"])
show_trace(result)
show("TOP RECOMMENDATION", result["recommendations"][0] if result["recommendations"] else "No safe result")

## Demo 2 — memory across turns

The second request does not repeat vegetarian. The agent reads the first turn's memory and uses it as a hard constraint.

In [ ]:
session = "notebook-demo-2"
clear_memory(session)
first = run_agent(session, "I'm vegetarian.")
show("TURN 1 FINAL RESULT", first["reply"])
show("MEMORY WRITE", get_memory(session))
second = run_agent(session, "Find me food in Jalandhar.")
show("TURN 2 USER INPUT", "Find me food in Jalandhar.")
show("MEMORY READ", second["trace"][1]["detail"])
show_trace(second)
print("diet used as hard constraint:", second["memory"]["diet"])

## Demo 3 — dietary conflict

A new non-vegetarian request conflicts with remembered vegetarian requirements. The agent refuses to recommend an incompatible result before calling restaurant tools.

In [ ]:
session = "notebook-demo-3"
clear_memory(session)
run_agent(session, "I'm vegetarian.")
conflict = run_agent(session, "I want a non-vegetarian restaurant in Jalandhar.")
show("USER INPUT", "I want a non-vegetarian restaurant in Jalandhar.")
show("MEMORY READ", get_memory(session))
show_trace(conflict)
print("recommendations returned:", len(conflict["recommendations"]))